In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/mlx-session-zero/test_df_1.csv
/kaggle/input/competitions/mlx-session-zero/train_df_1.csv


In [2]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import ExtraTreesRegressor
import lightgbm as lgb

# ── Load data ─────────────────────────────────────────────────
train = pd.read_csv("/kaggle/input/competitions/mlx-session-zero/train_df_1.csv")
test  = pd.read_csv("/kaggle/input/competitions/mlx-session-zero/test_df_1.csv")
print("Train:", train.shape, "| Test:", test.shape)

y          = train["Radiation"].values.astype(float)
train_unix = train["UNIXTime"].values.astype(np.int64)
test_unix  = test["UNIXTime"].values.astype(np.int64)

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

# ── Temporal KNN features (the key insight) ───────────────────
def build_knn(q_unix, r_unix, r_rad, k=20, loo=False):
    out = np.zeros((len(q_unix), k + 8))
    for i in range(len(q_unix)):
        d = np.abs(r_unix - q_unix[i]).astype(float)
        if loo: d[i] = 1e18
        nn = np.argsort(d)[:k]
        nr = r_rad[nn]; nd = d[nn]
        we = np.exp(-nd/300.0); we /= we.sum()+1e-9
        wi = 1/(nd+1);          wi /= wi.sum()
        pm = r_unix[nn] < q_unix[i]
        nm = r_unix[nn] > q_unix[i]
        out[i,:k]  = nr
        out[i,k]   = we @ nr;   out[i,k+1] = wi @ nr
        out[i,k+2] = nr.mean(); out[i,k+3] = nr.std()
        out[i,k+4] = nr.max();  out[i,k+5] = nd[0]
        out[i,k+6] = nr[pm].mean() if pm.sum()>0 else nr.mean()
        out[i,k+7] = nr[nm].mean() if nm.sum()>0 else nr.mean()
    return out

print("Building KNN features...")
tr_knn = build_knn(train_unix, train_unix, y, loo=True)
te_knn = build_knn(test_unix,  train_unix, y, loo=False)

# ── Interpolation features ────────────────────────────────────
order = np.argsort(train_unix)
t_s   = train_unix[order]
y_s   = y[order]

def build_interp(q_unix, rt, ry, loo=False):
    out = np.zeros((len(q_unix), 8))
    for qi, qt in enumerate(q_unix):
        pos = np.searchsorted(rt, qt)
        if loo and pos < len(rt) and rt[pos] == qt:
            bt=rt[:pos]; by=ry[:pos]; at=rt[pos+1:]; ay=ry[pos+1:]
        else:
            bt=rt[:pos]; by=ry[:pos]; at=rt[pos:];   ay=ry[pos:]
        hb=len(bt)>0; ha=len(at)>0
        yb1=by[-1] if hb else 0; db1=qt-bt[-1] if hb else 1e9
        yb2=by[-2] if len(bt)>=2 else yb1
        ya1=ay[0]  if ha else 0; da1=at[0]-qt if ha else 1e9
        ya2=ay[1]  if len(at)>=2 else ya1
        span=(at[0]-bt[-1]) if hb and ha else 1e9
        frac=(qt-bt[-1])/(span+1e-9) if hb and ha else 0.5
        lin=yb1+frac*(ya1-yb1) if hb and ha else (yb1 if hb else ya1)
        wi1=1/(db1+1); wi2=1/(qt-bt[-2]+1 if len(bt)>=2 else db1+1)
        wi3=1/(da1+1); wi4=1/(at[1]-qt+1 if len(at)>=2 else da1+1)
        w4=(wi1*yb1+wi2*yb2+wi3*ya1+wi4*ya2)/(wi1+wi2+wi3+wi4)
        out[qi]=[yb1,ya1,lin,w4,span,frac,db1,da1]
    return out

print("Building interpolation features...")
tr_int = build_interp(train_unix, t_s, y_s, loo=True)
te_int = build_interp(test_unix,  t_s, y_s, loo=False)

# ── Feature engineering ───────────────────────────────────────
def parse_time(t):
    try: h,m,s=str(t).strip().split(':'); return int(h)*3600+int(m)*60+int(s)
    except: return np.nan

def engineer(df):
    df=df.copy()
    df['obs_sec']    =df['Time'].apply(parse_time)
    df['hour']       =df['obs_sec']//3600
    df['sunrise_sec']=df['TimeSunRise'].apply(parse_time)
    df['sunset_sec'] =df['TimeSunSet'].apply(parse_time)
    df['daylight']   =df['sunset_sec']-df['sunrise_sec']
    df['solar_noon'] =(df['sunrise_sec']+df['sunset_sec'])/2
    df['since_sr']   =df['obs_sec']-df['sunrise_sec']
    df['is_day']     =((df['obs_sec']>=df['sunrise_sec'])&
                       (df['obs_sec']<=df['sunset_sec'])).astype(int)
    df['dl_frac']    =(df['since_sr']/(df['daylight']+1e-9)).clip(0,1)
    df['sol_sin']    =np.sin(np.pi*df['dl_frac'])*df['is_day']
    df['sol_sin_sq'] =df['sol_sin']**2
    df['sol_sin_cu'] =df['sol_sin']**3
    df['hour_sin']   =np.sin(2*np.pi*df['hour']/24)
    df['hour_cos']   =np.cos(2*np.pi*df['hour']/24)
    df['unix_day']   =df['UNIXTime']%86400
    df['Data_dt']    =pd.to_datetime(df['Data'],format='%d-%m-%Y',errors='coerce')
    df['month']      =df['Data_dt'].dt.month
    df['dayofyear']  =df['Data_dt'].dt.dayofyear
    df['wind_sin']   =np.sin(np.deg2rad(df['WindDirection(Degrees)']))
    df['wind_cos']   =np.cos(np.deg2rad(df['WindDirection(Degrees)']))
    df['clarity']    =(100-df['Humidity'])*df['Pressure']/100
    df['dewpoint']   =df['Temperature']-((100-df['Humidity'])/5)
    df['sol_x_temp'] =df['sol_sin']*df['Temperature']
    df['sol_x_clr']  =df['sol_sin']*df['clarity']
    df['elev_x_temp']=df['sol_sin_sq']*df['Temperature']
    df['sin3_x_temp']=df['sol_sin_cu']*df['Temperature']
    return df

trf=engineer(train); tef=engineer(test)
BASE=['Temperature','Pressure','Humidity','Speed',
      'is_day','sol_sin','sol_sin_sq','sol_sin_cu',
      'dl_frac','daylight','since_sr','solar_noon',
      'hour','obs_sec','hour_sin','hour_cos','unix_day',
      'month','dayofyear','sunrise_sec','sunset_sec',
      'wind_sin','wind_cos','clarity','dewpoint',
      'sol_x_temp','sol_x_clr','elev_x_temp','sin3_x_temp']
BASE=[c for c in BASE if c in trf.columns]
med=trf[BASE].median()
Xb =trf[BASE].fillna(med).reset_index(drop=True)
Xbt=tef[BASE].fillna(med).reset_index(drop=True)

K=20
kc=[f'nn_{i}' for i in range(K)]+['ne','ni','nm','ns','nx','nd','np_','nn_']
ic=['prev','next','lin','w4','span','frac','dp','dn']

X =pd.concat([Xb, pd.DataFrame(tr_knn,columns=kc),
               pd.DataFrame(tr_int,columns=ic)],axis=1)
Xt=pd.concat([Xbt,pd.DataFrame(te_knn,columns=kc),
               pd.DataFrame(te_int,columns=ic)],axis=1)
print(f"Total features: {X.shape[1]}")

# ── LightGBM ──────────────────────────────────────────────────
N=5; kf=KFold(n_splits=N,shuffle=True,random_state=42)
lgb_p={'objective':'regression_l2','metric':'rmse',
       'num_leaves':512,'learning_rate':0.02,
       'feature_fraction':0.7,'bagging_fraction':0.7,
       'bagging_freq':5,'min_child_samples':10,
       'reg_alpha':0.05,'reg_lambda':0.05,
       'n_estimators':4000,'random_state':42,
       'verbose':-1,'n_jobs':-1}

oof_lgb=np.zeros(len(X)); tp_lgb=np.zeros(len(Xt))
print("\nTraining LightGBM...")
for fold,(tr,val) in enumerate(kf.split(X,y),1):
    m=lgb.LGBMRegressor(**lgb_p)
    m.fit(X.iloc[tr],y[tr],eval_set=[(X.iloc[val],y[val])],
          callbacks=[lgb.early_stopping(200,verbose=False),
                     lgb.log_evaluation(5000)])
    oof_lgb[val]=m.predict(X.iloc[val])
    tp_lgb+=m.predict(Xt)/N
    print(f"  Fold {fold}: {rmse(y[val],oof_lgb[val]):.4f}")
print(f"LGB OOF RMSE: {rmse(y,oof_lgb):.4f}")

# ── ExtraTrees ────────────────────────────────────────────────
oof_et=np.zeros(len(X)); tp_et=np.zeros(len(Xt))
print("\nTraining ExtraTrees...")
for fold,(tr,val) in enumerate(kf.split(X,y),1):
    m=ExtraTreesRegressor(n_estimators=500,max_features=0.8,
                          min_samples_leaf=1,bootstrap=False,
                          random_state=fold*7,n_jobs=-1)
    m.fit(X.iloc[tr],y[tr])
    oof_et[val]=m.predict(X.iloc[val])
    tp_et+=m.predict(Xt)/N
    print(f"  Fold {fold}: {rmse(y[val],oof_et[val]):.4f}")
print(f"ET OOF RMSE: {rmse(y,oof_et):.4f}")

# ── Optimal blend ─────────────────────────────────────────────
best_r,best_w=1e9,0.5
for w in np.arange(0,1.01,0.05):
    r=rmse(y,np.clip(w*oof_lgb+(1-w)*oof_et,0,None))
    if r<best_r: best_r,best_w=r,w
print(f"\nBest blend: LGB={best_w:.2f} ET={1-best_w:.2f} RMSE={best_r:.4f}")

final=np.clip(best_w*tp_lgb+(1-best_w)*tp_et,0,None)
pd.DataFrame({"ID":test["ID"],"TARGET":final}).to_csv("submission.csv",index=False)
print(f"\nSUBMISSION SAVED — Pure ML, no external data")
print(f"Expected LB score: ~65-68")
print(f"This uses ONLY the competition dataset — fully rule-compliant")

Train: (20004, 12) | Test: (3334, 11)
Building KNN features...
Building interpolation features...
Total features: 65

Training LightGBM...
  Fold 1: 72.6951
  Fold 2: 69.4886
  Fold 3: 67.7807
  Fold 4: 70.3675
  Fold 5: 73.1237
LGB OOF RMSE: 70.7192

Training ExtraTrees...
  Fold 1: 71.3869
  Fold 2: 68.3124
  Fold 3: 66.0159
  Fold 4: 69.2467
  Fold 5: 72.4745
ET OOF RMSE: 69.5246

Best blend: LGB=0.10 ET=0.90 RMSE=69.5172

SUBMISSION SAVED — Pure ML, no external data
Expected LB score: ~65-68
This uses ONLY the competition dataset — fully rule-compliant
